In [1]:
# 03_fairness_metrics.ipynb

# %% [markdown]
# # Cross-Domain AI Governance Gap Analysis
# ## 03. Unified Baseline & Fairness Metrics
#
# Train one unified baseline (XGBoost) on every dataset and compute a common set of
# group-fairness metrics, so credit and insurance domains are directly comparable.
# - metrics: Statistical Parity Difference, Disparate Impact Ratio,
#   Equal Opportunity Difference, Average Odds Difference
# - each dataset uses a single binary sensitive attribute; Porto Seguro uses the
#   representative proxy ps_ind_04_cat (other proxies analyzed separately as a supplement)
#
# Two comparability fixes vs a naive 0.5 threshold:
#   (1) apply scale_pos_weight so imbalanced datasets don't collapse to all-negative
#   (2) set the decision threshold per dataset to the training positive rate (prevalence),
#       so "predicted positive" is meaningful across domains with different base rates

# %%
# Imports and configuration
import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")

RANDOM_STATE = 42
REPRESENTATIVE_PROXY = "ps_ind_04_cat"  # primary sensitive attribute for Porto Seguro
os.makedirs(TAB_DIR, exist_ok=True)

# %%
# Load metadata and define the single binary sensitive attribute per dataset
with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json")) as f:
    META = json.load(f)

# (display_name, key, primary sensitive column, privileged group value)
# privileged=None -> use the most frequent value as the reference group (for proxies)
DATASETS = [
    ("German Credit",        "german",               "sex",                "male"),
    ("Credit Card Approval", "credit_card_approval",  "sex",                "male"),
    ("Medical Cost",         "insurance",             "sex",                "male"),
    ("Porto Seguro",         "porto_seguro",          REPRESENTATIVE_PROXY, None),
]

def load_split(key):
    train = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_train.parquet"))
    test = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_test.parquet"))
    return train, test

# %%
# Build X, y by excluding target(s) and all sensitive columns from the feature matrix
def get_Xy(df, key):
    meta = META[key]
    target_col = meta["target_col"]
    sensitive_cols = meta["sensitive_cols"]
    aux_cols = [c for c in ["target_60dpd", "target_30dpd"] if c in df.columns and c != target_col]
    drop = [target_col] + sensitive_cols + aux_cols
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    y = df[target_col].astype(int)
    return X, y

# %%
# Group-fairness metric definitions.
# priv_mask / unpriv_mask are boolean arrays over the test set.
# Positive prediction here means "predicted high-risk (label = 1)".
def fairness_metrics(y_true, y_pred, priv_mask, unpriv_mask):
    def rates(mask):
        yt = y_true[mask]
        yp = y_pred[mask]
        ppr = yp.mean() if len(yp) else np.nan                  # positive prediction rate
        pos = yt == 1
        tpr = yp[pos].mean() if pos.sum() else np.nan           # true positive rate
        neg = yt == 0
        fpr = yp[neg].mean() if neg.sum() else np.nan           # false positive rate
        return ppr, tpr, fpr

    ppr_p, tpr_p, fpr_p = rates(priv_mask)
    ppr_u, tpr_u, fpr_u = rates(unpriv_mask)

    spd = ppr_u - ppr_p                                          # statistical parity diff
    di = (ppr_u / ppr_p) if ppr_p not in (0, np.nan) else np.nan # disparate impact ratio
    eod = tpr_u - tpr_p                                          # equal opportunity diff
    aod = 0.5 * ((fpr_u - fpr_p) + (tpr_u - tpr_p))              # average odds diff

    return {
        "StatParityDiff": spd,
        "DisparateImpact": di,
        "EqualOppDiff": eod,
        "AvgOddsDiff": aod,
    }

# %%
# Train unified baseline on each dataset and compute fairness metrics on the primary attribute
rows = []
pred_store = {}

for display_name, key, sens_col, privileged in DATASETS:
    train, test = load_split(key)
    X_train, y_train = get_Xy(train, key)
    X_test, y_test = get_Xy(test, key)
    X_test = X_test[X_train.columns]

    # Imbalance correction (identical in form across datasets -> still comparable)
    n_pos = int(y_train.sum())
    n_neg = int((y_train == 0).sum())
    spw = (n_neg / n_pos) if n_pos > 0 else 1.0

    model = XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, scale_pos_weight=spw,
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
    )
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]

    # Prevalence-based threshold: label the top (train positive rate) share as positive
    prevalence = float(y_train.mean())
    thr = np.quantile(proba, 1.0 - prevalence)
    y_pred = (proba >= thr).astype(int)

    # Define privileged / unprivileged groups on the primary sensitive attribute
    s = test[sens_col]
    if privileged is None:
        privileged = s.mode(dropna=True).iloc[0]
    priv_mask = (s == privileged).values
    unpriv_mask = (s != privileged).values & s.notna().values

    fm = fairness_metrics(y_test.values, y_pred, priv_mask, unpriv_mask)
    fm.update({
        "domain": META[key].get("domain", ""),
        "dataset": display_name,
        "sensitive": sens_col,
        "privileged": str(privileged),
        "threshold": round(thr, 4),
        "pred_pos_rate": round(float(y_pred.mean()), 4),
        "AUC": roc_auc_score(y_test.values, proba),
        "PR_AUC": average_precision_score(y_test.values, proba),
    })
    rows.append(fm)

    # Keep predictions (with sensitive cols) for the supplementary proxy analysis
    keep = list(dict.fromkeys([c for c in META[key]["sensitive_cols"]] + [sens_col]))
    pred_df = test[keep].copy()
    pred_df["y_true"] = y_test.values
    pred_df["proba"] = proba
    pred_df["y_pred"] = y_pred
    pred_store[key] = pred_df

    print(f"[{display_name:>22}] AUC={fm['AUC']:.3f}  thr={thr:.3f}  "
          f"predPos={fm['pred_pos_rate']:.3f}  "
          f"SPD={fm['StatParityDiff']:+.3f}  DI={fm['DisparateImpact']:.3f}  "
          f"EOD={fm['EqualOppDiff']:+.3f}  AOD={fm['AvgOddsDiff']:+.3f}")

# %%
# Primary fairness table
fair_df = pd.DataFrame(rows)[[
    "domain", "dataset", "sensitive", "privileged", "threshold", "pred_pos_rate",
    "AUC", "PR_AUC", "StatParityDiff", "DisparateImpact", "EqualOppDiff", "AvgOddsDiff",
]].round(4)
print(fair_df.to_string(index=False))
fair_df.to_csv(os.path.join(TAB_DIR, "03_fairness_primary.csv"), index=False)

# %%
# Supplementary analysis: Porto Seguro remaining proxy variables.
# Compute the same metrics using each additional proxy as the sensitive attribute,
# to check whether the representative proxy's result is typical or an outlier.
porto_key = "porto_seguro"
porto_test = load_split(porto_key)[1]
porto_pred = pred_store[porto_key]

proxy_cols = [c for c in META[porto_key]["sensitive_cols"]]
supp_rows = []
for pc in proxy_cols:
    s = porto_test[pc]
    if s.nunique(dropna=True) < 2:
        continue
    privileged = s.mode(dropna=True).iloc[0]
    priv_mask = (s == privileged).values
    unpriv_mask = (s != privileged).values & s.notna().values

    fm = fairness_metrics(
        porto_pred["y_true"].values, porto_pred["y_pred"].values,
        priv_mask, unpriv_mask
    )
    fm.update({"proxy": pc, "privileged": str(privileged),
               "is_representative": pc == REPRESENTATIVE_PROXY})
    supp_rows.append(fm)

supp_df = pd.DataFrame(supp_rows)[[
    "proxy", "privileged", "is_representative",
    "StatParityDiff", "DisparateImpact", "EqualOppDiff", "AvgOddsDiff",
]].round(4)
print(supp_df.to_string(index=False))
supp_df.to_csv(os.path.join(TAB_DIR, "03_porto_proxy_supplement.csv"), index=False)

[         German Credit] AUC=0.802  thr=0.503  predPos=0.300  SPD=+0.030  DI=1.103  EOD=+0.025  AOD=+0.006
[  Credit Card Approval] AUC=0.690  thr=0.563  predPos=0.118  SPD=-0.035  DI=0.753  EOD=-0.048  AOD=-0.039
[          Medical Cost] AUC=0.552  thr=0.548  predPos=0.251  SPD=+0.005  DI=1.018  EOD=+0.069  AOD=+0.027
[          Porto Seguro] AUC=0.556  thr=0.582  predPos=0.036  SPD=+0.001  DI=1.022  EOD=+0.001  AOD=+0.001
   domain              dataset     sensitive privileged  threshold  pred_pos_rate    AUC  PR_AUC  StatParityDiff  DisparateImpact  EqualOppDiff  AvgOddsDiff
   credit        German Credit           sex       male     0.5026         0.3000 0.8016  0.6262          0.0298           1.1026        0.0255       0.0065
   credit Credit Card Approval           sex       male     0.5632         0.1180 0.6904  0.2670         -0.0350           0.7528       -0.0482      -0.0392
insurance         Medical Cost           sex       male     0.5485         0.2507 0.5523  0.3008     